# FolkFusion — Colab Model Server

This runs `instruct-pix2pix` on a free T4 GPU and exposes it as a public API that your local Flask app calls.

**Steps:**
1. Make sure Runtime → Change runtime type → T4 GPU is selected
2. Run Cell 1 (install, ~2 min)
3. Run Cell 2 (load model, ~3 min)
4. Copy the `gradio.live` URL printed at the end
5. Paste it into your `.env` as `COLAB_URL=https://xxxx.gradio.live`
6. Restart your Flask app

In [ ]:
# Cell 1 — Install dependencies (~2 min)
!pip install -q diffusers transformers accelerate gradio torch

In [ ]:
# Cell 2 — Load model and start server (~3 min first time)
import torch
from diffusers import StableDiffusionInstructPix2PixPipeline, EulerAncestralDiscreteScheduler
from PIL import Image
import gradio as gr

print('Loading instruct-pix2pix model...')
pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    'timbrooks/instruct-pix2pix',
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to('cuda')
print('Model loaded!')

def transform(image, prompt, image_guidance, guidance, steps):
    result = pipe(
        prompt,
        image=image,
        num_inference_steps=int(steps),
        image_guidance_scale=float(image_guidance),
        guidance_scale=float(guidance),
    ).images[0]
    return result

iface = gr.Interface(
    fn=transform,
    inputs=[
        gr.Image(type='pil', label='Input image'),
        gr.Textbox(label='Style prompt'),
        gr.Slider(1.0, 2.5, value=1.5, step=0.1, label='Image guidance'),
        gr.Slider(1.0, 15.0, value=7.5, step=0.5, label='Prompt guidance'),
        gr.Slider(10, 50, value=25, step=1, label='Steps'),
    ],
    outputs=gr.Image(type='pil', label='Result'),
    title='FolkFusion Model Server',
)

iface.launch(share=True)